<div align="center">

<img src="https://raw.githubusercontent.com/ductho-le/WaveDL/main/logos/wavedl_logo.png" alt="WaveDL Logo" width="450">

# Interactive Demo

**Experience WaveDL in action — no installation required**

[![GitHub](https://img.shields.io/badge/⭐_Star_on_GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/ductho-le/WaveDL)
[![License](https://img.shields.io/badge/License-MIT-F97316?style=for-the-badge)](https://github.com/ductho-le/WaveDL/blob/main/LICENSE)
[![Python](https://img.shields.io/badge/Python-3.11+-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://www.python.org/)

</div>

---

## 🎯 What You'll Do

1. **Setup** — Clone repo & install dependencies
2. **Explore** — Visualize Lamb wave dispersion data
3. **Infer** — Run pre-trained CNN for material characterization
4. **Train** — (Optional) Quick training demo on synthetic data

---

### 📊 Demo Task: Material Characterization

This demo uses a **pre-trained CNN** to predict elastic properties from Lamb wave dispersion curves:

- **$h$** (mm) — Plate thickness
- **$\sqrt{E/\rho}$** (km/s) — Stiffness-to-density ratio  
- **$\nu$** — Lateral strain ratio (Poisson's ratio)

---

## 1️⃣ Setup Environment

In [ ]:
# Run setup safely from the repository, notebooks folder, or a fresh runtime.

import importlib
import os
from pathlib import Path
import site
import subprocess
import sys

from IPython.display import clear_output

working_dir = Path.cwd().resolve()
repo_dir = next(
    (candidate for candidate in (working_dir, *working_dir.parents)
     if (candidate / "src" / "wavedl").is_dir()
     and (candidate / "pyproject.toml").is_file()),
    None,
)
if repo_dir is None:
    repo_dir = working_dir / "WaveDL"
    if not repo_dir.exists():
        print("📦 Cloning WaveDL repository...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ductho-le/WaveDL.git", str(repo_dir)],
            check=True,
        )
    if not (repo_dir / "src" / "wavedl").is_dir():
        raise RuntimeError(f"{repo_dir} exists but is not a WaveDL checkout.")

print("📥 Installing WaveDL in the active notebook environment...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)], check=True)
os.chdir(repo_dir)

# Process the editable install in the already-running notebook kernel.
importlib.invalidate_caches()
site.main()

import torch
from wavedl import __version__

clear_output()
print(f"✅ Setup complete: WaveDL v{__version__} at {repo_dir}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
else:
    print("💻 Running on CPU")


---

## 2️⃣ Explore the Data

Each 500×500 pixel image encodes the wave propagation characteristics of a plate.

In [ ]:
# 📊 Visualize Dispersion Curves

import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({'font.size': 11, 'figure.facecolor': 'white'})

test_data = Path("examples/elasticity_prediction/Test_data_100.mat")

with h5py.File(test_data, 'r') as f:
    input_key = [k for k in f.keys() if 'input' in k.lower()][0]
    output_key = [k for k in f.keys() if 'output' in k.lower()][0]
    
    n_samples = f[input_key].shape[-1]
    input_shape = f[input_key].shape[:-1]
    n_targets = f[output_key].shape[0]
    
    np.random.seed(42)
    indices = np.random.choice(n_samples, 4, replace=False)
    samples = [f[input_key][:, :, i] for i in indices]
    targets = [f[output_key][:, i] for i in indices]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Sample Lamb Wave Dispersion Curves', fontsize=16, fontweight='bold', y=1.02)

param_names = [r'$h$', r'$\sqrt{E/\rho}$', r'$\nu$']
param_units = ['mm', 'km/s', '']

for idx, (ax, sample, target) in enumerate(zip(axes.flat, samples, targets)):
    im = ax.imshow(sample.T, aspect='auto', cmap='viridis', origin='lower')
    ax.set_xlabel('Frequency Index')
    ax.set_ylabel('Wavenumber Index')
    label = ' | '.join([f"{p}={v:.3f}{u}" for p, v, u in zip(param_names, target, param_units)])
    ax.set_title(f'Sample {indices[idx]+1}\n{label}', fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print(f"\n📦 Dataset: {n_samples} samples × {input_shape[0]}×{input_shape[1]} pixels")
print(f"🎯 Targets: {n_targets} parameters per sample")

---

## 3️⃣ Run Inference

Use the pre-trained CNN to predict material properties and generate diagnostic plots.

In [ ]:
# Run the pre-trained model. A failed command stops this cell immediately.

from IPython.display import Image, display
from pathlib import Path
import subprocess
import sys
from tempfile import mkdtemp

# Each run gets its own directory, so old plots cannot masquerade as new results.
results_root = Path("demo_results")
results_root.mkdir(exist_ok=True)
results_dir = Path(mkdtemp(prefix="run_", dir=results_root)).resolve()

print("🔄 Running inference on 100 samples...\n")
subprocess.run([
    sys.executable, "-m", "wavedl.test",
    "--checkpoint", "./examples/elasticity_prediction/best_checkpoint",
    "--data_path", "./examples/elasticity_prediction/Test_data_100.mat",
    "--param_names", r"$h$ (mm)", r"$\sqrt{E/\rho}$ (km/s)", r"$\nu$",
    "--plot", "--save_predictions",
    "--output_dir", str(results_dir),
], check=True)

plots = sorted(results_dir.glob("*.png"))
if not plots:
    raise RuntimeError(f"Inference produced no diagnostic plots in {results_dir}.")

print(f"\n✅ Generated {len(plots)} diagnostic plots in {results_dir}")
for plot_path in plots:
    print(f"\n📊 {plot_path.stem.replace('_', ' ').title()}")
    display(Image(filename=str(plot_path), width=700))


---

## 4️⃣ Quick Training Demo (Optional)

Try the training pipeline with synthetic data.

In [ ]:
# Generate synthetic data and exercise the training pipeline.
# These random inputs and targets are independent: this is a workflow smoke test,
# not a meaningful prediction benchmark.

import numpy as np
from pathlib import Path
import subprocess
import sys
from tempfile import mkdtemp

rng = np.random.default_rng(2025)
N_SAMPLES, IMG_SIZE, N_TARGETS = 200, 64, 3
X = rng.standard_normal((N_SAMPLES, IMG_SIZE, IMG_SIZE)).astype(np.float32)
y = rng.standard_normal((N_SAMPLES, N_TARGETS)).astype(np.float32)

training_root = Path("demo_training")
training_root.mkdir(exist_ok=True)
run_dir = Path(mkdtemp(prefix="run_", dir=training_root)).resolve()
training_data = run_dir / "demo_train.npz"
np.savez(training_data, input_train=X, output_train=y)

print(f"✅ Created: {N_SAMPLES} samples × {IMG_SIZE}×{IMG_SIZE} → {N_TARGETS} targets")
print("\n🏋️ Training for up to 5 epochs...\n")
subprocess.run([
    sys.executable, "-m", "wavedl.train",
    "--model", "cnn",
    "--data_path", str(training_data),
    "--batch_size", "32",
    "--epochs", "5",
    "--patience", "3",
    "--lr", "1e-3",
    "--workers", "0",
    "--output_dir", str(run_dir / "output"),
], check=True)

print(f"\n✅ Training complete! Results: {run_dir / 'output'}")


---

<br>

<div align="center">

<img src="https://raw.githubusercontent.com/ductho-le/WaveDL/main/logos/wavedl_logo.png" alt="WaveDL" width="150">

## 🎉 Demo Complete!

**You've experienced the full WaveDL workflow**

</div>

### 🚀 What's Next?

- 📖 **[Documentation](https://github.com/ductho-le/WaveDL)** — Full README & guides
- 🔧 **[Custom Models](https://github.com/ductho-le/WaveDL/blob/main/src/wavedl/models/_template.py)** — Add your architecture
- 📊 **[Data Format](https://github.com/ductho-le/WaveDL#-data-preparation)** — Prepare your dataset

### 💬 Get Help

- [Ask a Question](https://github.com/ductho-le/WaveDL/discussions) | [Report an Issue](https://github.com/ductho-le/WaveDL/issues)

---

<div align="center">

⭐ **Enjoying WaveDL?** [Star us on GitHub!](https://github.com/ductho-le/WaveDL)

*Made by [Ductho Le](https://github.com/ductho-le)*

</div>